# HR Analytics & Employee Attrition Analysis

## Project Overview
This project explores employee attrition patterns in the IBM HR Analytics Employee Attrition & Performance dataset. The goal is to use descriptive analysis and visualizations to identify patterns that HR teams may investigate further.

### Objectives
- Inspect dataset structure and quality.
- Calculate the overall employee attrition rate.
- Compare attrition rates across selected employee and workplace characteristics.
- Summarize evidence-based insights, recommendations, and limitations.

### Tools
Python, Pandas, Matplotlib, and Seaborn.

> **Interpretation note:** This is exploratory analysis. Differences between groups are associations and do not establish that a characteristic causes attrition.

## 1. Dataset Overview

Use the original CSV file from the IBM HR Analytics Employee Attrition & Performance dataset. For a portable project, place the CSV in a `data` folder beside this notebook, named `WA_Fn-UseC_-HR-Employee-Attrition.csv`. You can also place the ZIP file in the same folder as this notebook; the loading cell checks common filenames.

#### Import libraries

In [ ]:
from pathlib import Path
import zipfile

#### Configure visualization defaults

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#### Locate and load the dataset

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

#### Preview the dataset

In [ ]:
# Look for the CSV in common project locations first.
csv_candidates = [
    Path("data/WA_Fn-UseC_-HR-Employee-Attrition.csv"),
    Path("WA_Fn-UseC_-HR-Employee-Attrition.csv"),
    Path("data/HR-Employee-Attrition.csv"),
    Path("HR-Employee-Attrition.csv"),
]

In [ ]:
csv_path = next((path for path in csv_candidates if path.exists()), None)

In [ ]:
# If the CSV is not extracted, try common ZIP filenames in the project folder.
if csv_path is not None:
    df = pd.read_csv(csv_path)
else:
    zip_candidates = [
        Path("archive (4).zip"),
        Path("data/archive (4).zip"),
        Path("hr-analytics.zip"),
    ]
    zip_path = next((path for path in zip_candidates if path.exists()), None)

    if zip_path is None:
        raise FileNotFoundError(
            "Dataset not found. Place the dataset CSV at "
            "'data/WA_Fn-UseC_-HR-Employee-Attrition.csv' beside this notebook, "
            "or place the dataset ZIP in the notebook folder and update zip_candidates."
        )

    with zipfile.ZipFile(zip_path) as zf:
        csv_members = [name for name in zf.namelist() if name.lower().endswith(".csv")]
        if not csv_members:
            raise FileNotFoundError("The ZIP file does not contain a CSV file.")
        # This dataset ZIP is expected to contain the HR attrition CSV.
        preferred = [name for name in csv_members if "attrition" in Path(name).name.lower()]
        member = preferred[0] if preferred else csv_members[0]
        with zf.open(member) as file:
            df = pd.read_csv(file)

In [ ]:
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 2. Data Quality Checks

Check data types, missing values, duplicate rows, and duplicate employee identifiers before interpreting the data.

#### Review dataset structure

In [ ]:
print("Dataset information:")
df.info()

#### Check missing values

In [ ]:
print("\\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing values").head(10))

#### Check duplicate records

In [ ]:
print(f"Total missing values: {df.isna().sum().sum():,}")
print(f"Duplicate rows: {df.duplicated().sum():,}")

#### Review summary statistics

In [ ]:
if "EmployeeNumber" in df.columns:
    print(f"Duplicate EmployeeNumber values: {df['EmployeeNumber'].duplicated().sum():,}")

In [ ]:
print("\\nNumerical summary:")
display(df.describe().T)

In [ ]:
print("\\nCategorical summary:")
display(df.describe(include="object").T)

## 3. Overall Attrition

Calculate the number and percentage of employees who left and stayed, then visualize the overall distribution.

#### Count employees by attrition status

In [ ]:
attrition_counts = df["Attrition"].value_counts().reindex(["No", "Yes"], fill_value=0)
total_employees = len(df)
attrition_rate = (df["Attrition"].eq("Yes").mean() * 100)

#### Calculate the overall attrition rate

In [ ]:
print("Employee counts by attrition status:")
display(attrition_counts.to_frame("Employees"))
print(f"Overall attrition rate: {attrition_rate:.2f}%")

#### Plot overall attrition

In [ ]:
ax = sns.countplot(data=df, x="Attrition", order=["No", "Yes"])
ax.set(title="Employee Attrition Distribution", xlabel="Attrition status", ylabel="Number of employees")
for bar in ax.patches:
    ax.annotate(f"{int(bar.get_height()):,}",
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha="center", va="bottom", xytext=(0, 4), textcoords="offset points")
plt.tight_layout()
plt.show()

## 4. Attrition by Department

Compare the share of employees leaving within each department. These percentages use each department's own employee count as the denominator.

#### Calculate department attrition rates

In [ ]:
department_attrition = (
    df.groupby("Department")["Attrition"]
      .apply(lambda values: values.eq("Yes").mean() * 100)
      .sort_values(ascending=False)
)
display(department_attrition.round(2).to_frame("Attrition rate (%)"))

#### Plot department attrition rates

In [ ]:
ax = department_attrition.plot(kind="bar")
ax.set(title="Employee Attrition Rate by Department", xlabel="Department", ylabel="Attrition rate (%)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 5. Attrition by Overtime

Compare attrition rates for employees who work overtime and those who do not.

#### Calculate overtime attrition rates

In [ ]:
overtime_attrition = (
    df.groupby("OverTime")["Attrition"]
      .apply(lambda values: values.eq("Yes").mean() * 100)
      .reindex(["Yes", "No"])
)
display(overtime_attrition.round(2).to_frame("Attrition rate (%)"))

#### Plot overtime attrition rates

In [ ]:
ax = overtime_attrition.plot(kind="bar")
ax.set(title="Employee Attrition Rate by Overtime", xlabel="Overtime", ylabel="Attrition rate (%)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 6. Attrition by Job Role

Compare the percentage of employees leaving within each job role. Smaller job groups may have less stable rates, so interpret the percentages alongside group sizes when making decisions.

#### Summarize attrition by job role

In [ ]:
jobrole_summary = (
    df.groupby("JobRole")
      .agg(
          employees=("Attrition", "size"),
          leavers=("Attrition", lambda values: values.eq("Yes").sum()),
          attrition_rate=("Attrition", lambda values: values.eq("Yes").mean() * 100),
      )
      .sort_values("attrition_rate", ascending=False)
)
display(jobrole_summary.round({"attrition_rate": 2}))

#### Plot job-role attrition rates

In [ ]:
ax = jobrole_summary["attrition_rate"].plot(kind="bar", figsize=(10, 5))
ax.set(title="Employee Attrition Rate by Job Role", xlabel="Job role", ylabel="Attrition rate (%)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 7. Attrition by Age Group

Create age bands and compare attrition rates. The bins below represent ages 18–25, 26–35, 36–45, 46–55, and 56+.

#### Create age groups

In [ ]:
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[17, 25, 35, 45, 55, np.inf],
    labels=["18-25", "26-35", "36-45", "46-55", "56+"],
    include_lowest=True
)

#### Summarize attrition by age group

In [ ]:
age_group_summary = (
    df.groupby("AgeGroup", observed=False)
      .agg(
          employees=("Attrition", "size"),
          leavers=("Attrition", lambda values: values.eq("Yes").sum()),
          attrition_rate=("Attrition", lambda values: values.eq("Yes").mean() * 100),
      )
)
display(age_group_summary.round({"attrition_rate": 2}))

#### Plot age-group attrition rates

In [ ]:
ax = age_group_summary["attrition_rate"].plot(kind="bar")
ax.set(title="Employee Attrition Rate by Age Group", xlabel="Age group", ylabel="Attrition rate (%)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 8. Monthly Income by Attrition Status

Compare the distribution of monthly income for employees who stayed and those who left. The box plot complements mean and median summaries by showing spread and potential outliers.

#### Summarize monthly income

In [ ]:
income_summary = df.groupby("Attrition")["MonthlyIncome"].agg(
    count="count", mean="mean", median="median"
).round(2)
display(income_summary)

#### Plot monthly income by attrition status

In [ ]:
ax = sns.boxplot(data=df, x="Attrition", y="MonthlyIncome", order=["No", "Yes"])
ax.set(title="Monthly Income by Attrition Status", xlabel="Attrition status", ylabel="Monthly income")
plt.tight_layout()
plt.show()

## 9. Job Satisfaction, Work-Life Balance, Tenure, and Salary Hike

Compare summary statistics between employees who stayed and employees who left. These are descriptive comparisons and should not be interpreted as causal effects.

In [ ]:
comparison_columns = [
    "JobSatisfaction",
    "WorkLifeBalance",
    "YearsAtCompany",
    "PercentSalaryHike",
]
for column in comparison_columns:
    print(f"\\n{column} by attrition:")
    display(df.groupby("Attrition")[column].agg(
        count="count", mean="mean", median="median"
    ).round(2))

## 10. Attrition by Business Travel, Stock Options, Gender, and Job Involvement

For categorical factors, calculate attrition rate within each category. Keep the category names visible so the results are easy to interpret.

#### Define attrition-rate function

In [ ]:
def attrition_rate_by(column):
    return (
        df.groupby(column, dropna=False)["Attrition"]
          .apply(lambda values: values.eq("Yes").mean() * 100)
          .sort_values(ascending=False)
          .round(2)
          .to_frame("Attrition rate (%)")
    )

#### Calculate rates for selected categories

In [ ]:
for column in ["BusinessTravel", "StockOptionLevel", "Gender", "JobInvolvement"]:
    print(f"\\nAttrition by {column}:")
    display(attrition_rate_by(column))

## 11. Attrition by Performance Rating

Check whether attrition rates differ across performance rating groups.

In [ ]:
performance_attrition = (
    df.groupby("PerformanceRating")["Attrition"]
      .apply(lambda values: values.eq("Yes").mean() * 100)
      .round(2)
      .to_frame("Attrition rate (%)")
)
display(performance_attrition)

## 12. Business Insights

1. **Overall attrition:** 237 of 1,470 employees left, an overall attrition rate of approximately 16.12%.
2. **Overtime:** Employees working overtime had an attrition rate of approximately 30.53%, compared with 10.44% among employees not working overtime.
3. **Job role:** Sales Representatives had the highest observed job-role attrition rate (approximately 39.76%); Research Directors had the lowest (approximately 2.50%). Consider group sizes when interpreting rates.
4. **Department:** Sales had the highest department attrition rate (approximately 20.63%), followed by Human Resources (approximately 19.05%).
5. **Age:** Employees aged 18–25 had the highest age-group attrition rate (approximately 35.77%).
6. **Monthly income:** Employees who left had lower average monthly income (approximately 4,787.09) than employees who stayed (approximately 6,832.74).
7. **Job satisfaction:** Employees who left had a lower average job satisfaction score (approximately 2.47) than employees who stayed (approximately 2.78).
8. **Tenure:** Median company tenure was 3 years among employees who left and 6 years among employees who stayed.
9. **Business travel:** Employees who travelled frequently had an attrition rate of approximately 24.91%, compared with 8.00% among non-travellers.
10. **Job involvement:** Employees with a job involvement score of 1 had an attrition rate of approximately 33.73%, compared with 9.03% for score 4.
11. **Small differences:** Average salary hike percentages and attrition rates across performance ratings were very similar between groups.

> These results describe patterns in this dataset; they do not establish the causes of employee attrition.

## 13. Recommendations

1. **Review overtime workloads:** Examine overtime frequency, staffing levels, and employee feedback to understand the observed difference in attrition rates.
2. **Investigate Sales Representative retention:** Explore onboarding, career progression, compensation, workload, and manager support for this role.
3. **Support early-career employees:** Review onboarding, mentoring, training, and career development for younger employees.
4. **Review compensation fairly:** Compare compensation by role, experience, and other relevant factors before drawing conclusions from income differences.
5. **Strengthen employee engagement:** Use confidential feedback and follow-up analysis to better understand job satisfaction and involvement patterns.
6. **Review travel expectations:** Gather feedback from frequent business travellers about scheduling and workload.
7. **Monitor trends over time:** Track attrition by role, department, tenure, and overtime in future periods.

These are hypotheses and potential actions for investigation, not proof of causes. Validate them with additional organizational data and employee feedback before making decisions.

## 14. Conclusion

This exploratory analysis examined employee attrition in a dataset containing 1,470 employee records and 35 columns. The overall attrition rate was approximately 16.12%.

Higher observed attrition rates appeared among employees working overtime, Sales Representatives, employees aged 18–25, and employees with lower job involvement scores. Employees who left also had lower average monthly income, job satisfaction, and company tenure than employees who stayed.

Python and Pandas were used to inspect and summarize the data, while Matplotlib and Seaborn supported visualization. The findings can help identify areas for further HR investigation, but they do not prove why employees leave.

## 15. Limitations

1. **Dataset scope:** The analysis covers 1,470 records from one dataset and may not generalize to other organizations.
2. **Correlation versus causation:** Descriptive comparisons cannot establish why employees leave.
3. **Limited context:** The dataset does not capture all personal reasons, management practices, or workplace experiences that may influence attrition.
4. **No predictive model:** This project focuses on exploratory data analysis and does not build or validate a predictive model.
5. **Data currency:** The dataset may not reflect current workforce conditions.
6. **Group-size differences:** Some job roles or categories may contain fewer employees, making their rates less stable. Group sizes should be considered alongside percentages.
7. **Further validation:** HR teams should validate patterns using employee feedback and additional organizational data before acting on them.

## 16. Reproducibility Checklist

Before publishing:
- Run all cells from a fresh kernel, from top to bottom.
- Confirm that the dataset path works on your machine and is documented.
- Check all chart labels, calculations, and displayed outputs.
- Do not publish sensitive or restricted data.
- Add a README with the dataset source, project objectives, tools, and main findings.

**Dataset note:** Cite the original dataset provider and follow its terms of use when sharing or redistributing the data.